# WildfireSpreadTS -- CPU-only prep: download, preprocess, persist as kernel output

No GPU here on purpose -- everything in this notebook (download, unzip,
GeoTIFF->HDF5 conversion) is I/O/CPU-bound, and the previous combined run
burned ~6 hours of GPU-quota time sitting idle on a slow download. This run
writes the final HDF5 data to `/kaggle/working`, so the GPU training run
(separate notebook) can attach this kernel's own output as an input via
`kernel_sources` -- no dataset upload, no extra credentials needed.

Fix vs. the previous run: that one used `pip install -r requirements.txt`
(pinned 2023 versions) which failed a wheel build, silently swallowed by a
`| tail -20` in the log command -- the real error was never seen. This
version installs only the specific missing packages (matching what actually
worked in the diagnostic run) and does NOT truncate any error output.

In [ ]:
import os, subprocess, sys

os.system('df -h /kaggle/working /kaggle/temp 2>&1 || df -h /kaggle')

TEMP = '/kaggle/temp/wfts'
os.makedirs(TEMP, exist_ok=True)
OUT = '/kaggle/working/wfts_hdf5_dataset'
os.makedirs(OUT, exist_ok=True)
print('Scratch:', TEMP, ' Persisted output:', OUT, flush=True)

In [ ]:
# Clone repo
!git clone --depth 1 https://github.com/SebastianGer/WildfireSpreadTS.git /kaggle/working/wfts_repo

In [ ]:
# Patch: the repo (written ~2023) imports the PyTorch-internal name `T_co`
# from torch.utils.data.dataset, which newer PyTorch renamed to `_T_co`.
# Confirmed via the previous run's actual traceback -- not a guess. Patch
# every occurrence in the cloned copy rather than assuming it's only one file.
import subprocess as sp

grep = sp.run(['grep', '-rl', 'from torch.utils.data.dataset import T_co', '/kaggle/working/wfts_repo'],
              capture_output=True, text=True)
files = [f for f in grep.stdout.splitlines() if f]
print('Files needing the T_co -> _T_co patch:', files, flush=True)
for fpath in files:
    sp.run(['sed', '-i', 's/from torch.utils.data.dataset import T_co/from torch.utils.data.dataset import _T_co as T_co/', fpath], check=True)
print('Patched.', flush=True)

In [ ]:
# Install only what's actually missing -- Kaggle's image already ships torch,
# numpy, rasterio, gdal etc, which is why targeting the repo's full pinned
# requirements.txt broke last time. Do NOT pipe this through tail/truncate --
# if it fails, we need the real error, not a generic wrapper message.
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'lightning', 'h5py'],
    capture_output=True, text=True,
)
print(result.stdout[-3000:], flush=True)
print(result.stderr[-3000:], flush=True)
print('pip exit code:', result.returncode, flush=True)
if result.returncode != 0:
    raise RuntimeError('pip install failed -- see full output above, do not proceed silently.')

In [ ]:
# Confirm the CreateHDF5Dataset.py script's own imports actually resolve
# before spending hours on a download, in case it needs something else.
result = subprocess.run(
    [sys.executable, '-c', 'import sys; sys.path.insert(0, "/kaggle/working/wfts_repo/src"); '
     'import ast; ast.parse(open("/kaggle/working/wfts_repo/src/preprocess/CreateHDF5Dataset.py").read())'],
    capture_output=True, text=True,
)
print(result.stdout, result.stderr, flush=True)

# Actually try importing it for real (this will surface missing deps early)
result = subprocess.run(
    [sys.executable, '/kaggle/working/wfts_repo/src/preprocess/CreateHDF5Dataset.py', '--help'],
    capture_output=True, text=True, cwd='/kaggle/working/wfts_repo',
)
print('--- CreateHDF5Dataset.py --help ---', flush=True)
print(result.stdout, flush=True)
print(result.stderr, flush=True)
if result.returncode != 0:
    raise RuntimeError('CreateHDF5Dataset.py cannot even print --help -- missing dependency, fix before downloading 48GB.')
print('OK: preprocessing script imports cleanly.', flush=True)

In [ ]:
# Download (chunked, flushed progress every 2GB -- less chatty)
import time, urllib.request, hashlib

url = 'https://zenodo.org/records/8006177/files/WildfireSpreadTS.zip?download=1'
expected_md5 = 'dc1a04e63ccc70037b277d585b8fe761'
dest = f'{TEMP}/WildfireSpreadTS.zip'

t0 = time.time()
chunk_size = 1 << 20
report_every = 2000  # MB
downloaded = 0

with urllib.request.urlopen(url) as resp, open(dest, 'wb') as f:
    total = int(resp.headers.get('Content-Length', 0))
    print(f'Total size: {total / 1e9:.2f} GB', flush=True)
    last_report_mb = 0
    while True:
        chunk = resp.read(chunk_size)
        if not chunk:
            break
        f.write(chunk)
        downloaded += len(chunk)
        mb = downloaded // (1 << 20)
        if mb - last_report_mb >= report_every:
            elapsed = time.time() - t0
            rate = downloaded / elapsed / 1e6 if elapsed > 0 else 0
            pct = 100 * downloaded / total if total else 0
            print(f'  {mb} MB ({pct:.1f}%), {rate:.1f} MB/s, {elapsed:.0f}s elapsed', flush=True)
            last_report_mb = mb

print(f'Download complete: {downloaded / 1e9:.2f} GB in {time.time() - t0:.0f}s', flush=True)

h = hashlib.md5()
with open(dest, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
actual = h.hexdigest()
print('expected:', expected_md5, ' actual:', actual, flush=True)
assert actual == expected_md5, 'Checksum mismatch.'
print('OK: checksum verified.', flush=True)

In [ ]:
extract_dir = f'{TEMP}/extracted'
os.makedirs(extract_dir, exist_ok=True)
t0 = time.time()
rc = os.system(f'unzip -q {dest} -d {extract_dir}')
print(f'unzip exit code: {rc}, {time.time()-t0:.0f}s', flush=True)
if rc != 0:
    raise RuntimeError('unzip failed.')
os.system(f'du -sh {extract_dir}')
os.remove(dest)
print('Removed zip after extraction.', flush=True)

In [ ]:
# Preprocess to HDF5 -- write straight to /kaggle/working (the FINAL result
# we want to keep is comparatively small HDF5 files, not the raw GeoTIFFs)
t0 = time.time()
result = subprocess.run(
    [sys.executable, 'src/preprocess/CreateHDF5Dataset.py',
     '--data_dir', extract_dir, '--target_dir', OUT],
    cwd='/kaggle/working/wfts_repo', capture_output=True, text=True,
)
print(result.stdout[-5000:], flush=True)
print(result.stderr[-5000:], flush=True)
print(f'CreateHDF5Dataset exit code: {result.returncode}, {time.time()-t0:.0f}s', flush=True)
os.system(f'du -sh {OUT}')
if result.returncode != 0:
    raise RuntimeError('HDF5 preprocessing failed -- see output above.')

In [ ]:
# NOT calling `kaggle datasets create` here -- that would need Kaggle API
# credentials injected INTO this container, a new credential-handling
# problem worth avoiding. Instead: this kernel's own /kaggle/working output
# (already written above) can be attached directly to another kernel via
# that kernel's `kernel_sources` in kernel-metadata.json -- no separate
# dataset/auth step needed. Just report the final size so we know it fits
# under Kaggle's ~20GB persisted-output cap.
os.system(f'du -sh {OUT}')
os.system(f'find {OUT} -maxdepth 2 | head -20')
print('Done. Reference this kernel via "kernel_sources": ["muhammadalihashar/wildfirespreadts-prepare-real-data"] in the training kernel.', flush=True)